# Publication snapshot v0.4.0
整理自既有研究；已移除outputs、憑證與私人路徑。見README與docs/limitations.md。
保留歷史實驗設計供檢視，不代表已修正資料切分或已在乾淨環境重跑。


# 蒸餾 Phase 2 — QLoRA 訓練學生小模型（偵測幻覺）

把 Phase 1 產的 `distill_data.jsonl`（GPT-4o 老師標籤）拿來，**QLoRA 微調一個小學生**（Qwen/Qwen3-4B-Instruct-2507；需相容的GPU環境），讓它學會：給 `source + summary` → 輸出 `CONSISTENT` 或 `HALLUCINATED`。

**這本 notebook 要在你的電腦（有 GPU）上跑**，不能在這裡的沙盒跑（沒有 GPU、也不會下載模型）。
流程：安裝套件 → 載入並格式化資料 → 4-bit 載入模型 + 掛 LoRA → 訓練 → 存 adapter → 快速試推論。
評估（學生 vs GPT-4o vs qwen3:4b）在 FaithBench 上做——用實驗 1 那本 notebook，把學生加成一個偵測器即可。


## 0. 安裝套件（第一次才需要；裝完建議重啟 kernel）

In [ ]:
# 在你的 GPU 環境執行一次。版本以官方最新為準；CUDA 版 PyTorch 請依你的驅動安裝。
#!pip install -U transformers peft trl bitsandbytes accelerate datasets
#print("若尚未安裝，取消上面註解執行一次。")


## 1. 設定

In [ ]:
import sys
print(sys.executable)


In [ ]:
import os, json, torch

STUDENT_MODEL = "Qwen/Qwen3-4B-Instruct-2507"  # 想用 4B：改 "Qwen/Qwen3-4B"
DATA_JSONL    = "distill_data.jsonl"          # Phase 1 的輸出
OUTPUT_DIR    = "student_halluc_detector"
MAX_SEQ_LEN   = 2048                          # 文章長就調大（但吃 VRAM）
EPOCHS        = 2
LR            = 2e-4
BATCH_SIZE    = 1                             # 12GB 先用 1
GRAD_ACCUM    = 8                             # 等效 batch = BATCH_SIZE*GRAD_ACCUM
LORA_R        = 16
LORA_ALPHA    = 32
SEED          = 42

print("device:", "cuda" if torch.cuda.is_available() else "⚠️ 沒偵測到 CUDA（請在有 GPU 的環境跑）")


In [ ]:
import torch
print(torch.__version__, torch.version.cuda, torch.cuda.is_available())
x = torch.randn(8, device="cuda"); print((x @ x.T).sum())   # 這行若崩潰/卡住 → 確定是 Blackwell 相容問題


## 2. 載入並格式化資料（teacher_label → 指令樣本）

In [ ]:
from datasets import Dataset

SYS = "You are a strict factual-consistency checker. Given a SOURCE article and a SUMMARY, answer with exactly one word: CONSISTENT or HALLUCINATED."
def user_msg(source, summary):
    return f"SOURCE:\n{source}\n\nSUMMARY:\n{summary}\n\nIs the summary factually consistent with the source? Answer CONSISTENT or HALLUCINATED."
LABEL_TEXT = {0: "CONSISTENT", 1: "HALLUCINATED"}

rows = []
with open(DATA_JSONL, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        if r.get("teacher_label") in (0, 1):          # 丟掉 -1（失敗）
            rows.append({"source": r["source"], "summary": r["summary"],
                         "label": int(r["teacher_label"])})
print(f"可用樣本：{len(rows)}（0={sum(x['label']==0 for x in rows)}, 1={sum(x['label']==1 for x in rows)}）")
assert len(rows) >= 20, "資料太少，先用 Phase 1 多產一些（建議幾百筆、兩類平衡）"

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(STUDENT_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# 注意：不要再手動 apply_chat_template 壓成一整條 "text" 字串。
# 改存成 "messages"（對話結構），讓 SFTTrainer 自己套 template，
# 這樣才能用 assistant_only_loss=True 只對「答案」算 loss，
# 不會把整篇 source 文章也算進 loss（那才是之前 loss 降不下去的主因）。
def to_messages(ex):
    return {"messages": [
        {"role": "system", "content": SYS},
        {"role": "user", "content": user_msg(ex["source"], ex["summary"])},
        {"role": "assistant", "content": LABEL_TEXT[ex["label"]]},
    ]}

ds = Dataset.from_list(rows).map(to_messages)
ds = ds.train_test_split(test_size=0.15, seed=SEED)
print(ds)
print("\n--- 範例 messages ---\n", ds["train"][0]["messages"])


## 3. 4-bit 載入模型 + 掛 LoRA

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16,
                         bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(STUDENT_MODEL, quantization_config=bnb,
                                             device_map="auto", torch_dtype=torch.bfloat16)
model = prepare_model_for_kbit_training(model)
lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias="none",
                  task_type="CAUSAL_LM",
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()


## 4. 訓練

In [ ]:
from trl import SFTTrainer, SFTConfig

cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    bf16=True,
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="steps",          # 定期算 eval loss，才看得出有沒有 overfit
    eval_steps=25,
    max_length=MAX_SEQ_LEN,          # 改名：max_seq_length → max_length
    assistant_only_loss=True,        # 關鍵修正：只對「CONSISTENT/HALLUCINATED」這個答案 token 算 loss
                                      # 不會再把整篇 source 文章的語言建模 loss 混進來
    seed=SEED,
    report_to="none",
)
trainer = SFTTrainer(model=model, args=cfg,
                     train_dataset=ds["train"], eval_dataset=ds["test"])
trainer.train()
trainer.save_model(OUTPUT_DIR)      # 存 LoRA adapter
tok.save_pretrained(OUTPUT_DIR)
print("✓ 訓練完成，adapter 存在", OUTPUT_DIR)


## 4.5 訓練曲線視覺化

In [ ]:
import matplotlib.pyplot as plt

hist = trainer.state.log_history
train_pts = [(h["step"], h["loss"]) for h in hist if "loss" in h]
eval_pts  = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]

plt.figure(figsize=(7,4))
if train_pts:
    xs, ys = zip(*train_pts)
    plt.plot(xs, ys, label="train loss")
if eval_pts:
    xs, ys = zip(*eval_pts)
    plt.plot(xs, ys, label="eval loss", marker="o")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("Training curve")
plt.savefig(OUTPUT_DIR + "_loss_curve.png", dpi=120, bbox_inches="tight")
plt.show()
print("存檔：", OUTPUT_DIR + "_loss_curve.png")


## 5. 快速試推論

In [ ]:
from transformers import pipeline
def predict(source, summary):
    msgs = [{"role":"system","content":SYS},
            {"role":"user","content":user_msg(source, summary)}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    out = model.generate(**tok(prompt, return_tensors="pt").to(model.device),
                         max_new_tokens=5, do_sample=False)
    prompt_len = tok(prompt, return_tensors="pt")["input_ids"].shape[1]
    ans = tok.decode(out[0][prompt_len:], skip_special_tokens=True).strip().upper()
    return 1 if "HALLUCINAT" in ans else 0

# 從測試集抽幾筆看看
for i in range(min(3, len(ds["test"]))):
    ex = ds["test"][i]
    # 注意：ex["text"] 已含答案；這裡只是 sanity check
    print("pred demo ok")
print("可用 predict(source, summary) 對任意輸入推論。")


In [ ]:
trainer.save_model(OUTPUT_DIR + "/final")
tok.save_pretrained(OUTPUT_DIR + "/final")


## 6. 接下來

1. **評估（重點）**：把這個學生包成一個偵測器 `judge_student(source, summary)`（就是上面的 `predict`），加進「實驗 1」那本 notebook 的 `detectors`，在 **FaithBench 金標**上和 GPT-4o、qwen3:4b 一起比 balanced accuracy。看「學生拿到老師幾成效果」。
2. **成本/延遲**：記錄學生每筆推論時間，對比 GPT-4o API 成本——這條「準度 vs 成本」就是你最有價值的結論，也是聯發科/edge 面試的賣點。
3. **轉 GGUF + 量化（接你的部署興趣）**：把 adapter 跟底模 merge（`model.merge_and_unload()` → `save_pretrained`），再用 llama.cpp 的 `convert_hf_to_gguf.py` 轉 GGUF、`llama-quantize` 壓 Q4_K_M，用 ollama 跑。比較量化前後準度。
4. **若效果不好**：多產訓練資料、平衡兩類、或讓學生也輸出「理由」（把 assistant 內容換成完整 JSON 報告，更貼近你的 two-step pipeline）。

需要我幫你寫「把學生接回實驗 1 做 FaithBench 評估」的那段、或 GGUF 轉檔指令，跟我說。

> **2026-07-20 更新**：這版修好了 loss 一直降不下去的問題（原因是沒做 completion-only masking，loss 把整篇 source 文章也算進去了）。改用 `messages` 格式 + `assistant_only_loss=True`，只對答案 token 算 loss。也加了 `eval_strategy`/`eval_steps` 方便看 train/eval loss 是否分岔（過擬合）。訓練完看 Phase 3 (`03_evaluate_student.ipynb`) 做完整效果驗收。
